# Part 2 - 01 Data preparation for MobileNetV2

Same dataset, same loader and same split as Part 1, so both models are compared on the same test set.

## Imports and configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from neurovision.data import CLASSES, split_fingerprint
from neurovision.preprocessing import prepare_data, make_dataset, encoder
from neurovision.transfer import IMAGE_SIZE, build_augmentation, build_mobilenet_preprocessing

RAW_DIR = "../data/raw"

## Load, clean and split (same function and seed as Part 1)

In [ ]:
data = prepare_data(RAW_DIR, size=IMAGE_SIZE, seed=42)
X_train, y_train, p_train = data["train"]
X_val, y_val, p_val = data["val"]
X_test, y_test, p_test = data["test"]

print(X_train.shape, X_val.shape, X_test.shape)
print(X_train.dtype, X_train.min(), X_train.max())

## Split fingerprints (identical values in Part 1 and Part 2 prove the same split)

In [ ]:
for name, paths in [("train", p_train), ("val", p_val), ("test", p_test)]:
    print(name, split_fingerprint(paths, RAW_DIR))

## Leakage check and class proportions per split

In [ ]:
print(len(set(p_train) & set(p_val)), len(set(p_train) & set(p_test)), len(set(p_val) & set(p_test)))

for name, labels in [("train", y_train), ("val", y_val), ("test", y_test)]:
    values, counts = np.unique(labels, return_counts=True)
    print(name, dict(zip(values, np.round(counts / len(labels), 3))))

## Class weights computed on the training set only

In [ ]:
y_train_idx = encoder.transform(y_train)
weights = compute_class_weight("balanced", classes=np.arange(len(CLASSES)), y=y_train_idx)
class_weight = dict(enumerate(weights))
print(class_weight)

## MobileNetV2 preprocessing: pixels from [0, 255] to [-1, 1]

In [ ]:
batch = X_train[:8].astype("float32")
expected = preprocess_input(batch.copy())
actual = build_mobilenet_preprocessing()(batch).numpy()

print("Before:", batch.min(), batch.max())
print("After:", actual.min(), actual.max())
print("Max difference with preprocess_input:", np.abs(expected - actual).max())

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(X_train[0])
axes[0].set_title("Before: 0 to 255")
axes[0].axis("off")
axes[1].imshow((actual[0] + 1) / 2)
axes[1].set_title("After: -1 to 1 (rescaled for display)")
axes[1].axis("off")
plt.savefig("../reports/part2_preprocessing.png", dpi=150, bbox_inches="tight")
plt.show()

## Augmentation preview (applied to the training set only, inside the model)

In [ ]:
augmentation = build_augmentation()
sample = X_train[:1].astype("float32")

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(X_train[0])
axes[0].set_title("Original")
axes[0].axis("off")
for i in range(1, 5):
    augmented = augmentation(sample, training=True)[0].numpy().clip(0, 255).astype("uint8")
    axes[i].imshow(augmented)
    axes[i].set_title(f"Augmented {i}")
    axes[i].axis("off")
plt.savefig("../reports/part2_augmentation.png", dpi=150, bbox_inches="tight")
plt.show()

## Build the tf.data datasets (labels one-hot encoded)

In [ ]:
train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)

xb, yb = next(iter(train_ds))
print(xb.shape, xb.dtype, yb.shape)